Redis 在后端面试里属于“基础 + 场景 + 工程实践”三合一的高频模块。

如果只是背 Redis 有 String、Hash、List、Set、ZSet，面试深度是不够的。真正比较好的回答方式是：

第一，先讲 Redis 底层数据结构。
比如 String 的 SDS、Hash 的 listpack/hashtable、ZSet 的 skiplist、Set 的 intset/hashtable、List 的 quicklist/listpack。

第二，讲 Redis 为什么快。
纯内存、高效数据结构、单线程命令执行、IO 多路复用、Redis 6 之后网络 IO 多线程。

第三，讲缓存常见问题。
缓存穿透、缓存击穿、缓存雪崩、缓存预热、缓存淘汰、BigKey、HotKey。

第四，讲一致性和分布式锁。
数据库和缓存双写一致性、先更新 DB 再删除缓存、延时双删、MQ/binlog 补偿、SET NX EX、Lua 原子解锁、锁续期、Redlock 争议。

第五，讲 Go 项目怎么落地。
在 Go 中最常用的是 `go-redis` 客户端，要重点说清楚 context 超时、连接池配置、pipeline、Lua、singleflight、本地缓存、错误处理、重试和降级。

面试里我会把 Redis 总结成一句话：

Redis 不是单纯的缓存组件，而是一个基于内存、高效数据结构和单线程事件模型构建的高性能 KV 系统；在 Go 项目里，使用 Redis 的关键不是会调用 Get/Set，而是要处理好超时、连接池、缓存一致性、热点保护、分布式锁和降级兜底。

## 8.1 Go 项目中如何使用 Redis

### 8.1.1 Q：Go 项目里通常怎么接入 Redis？
Go 项目里最常用的 Redis 客户端是 `github.com/redis/go-redis/v9`。

它的核心特点是：

1. 支持 context
   每次 Redis 调用都可以带超时、取消和 trace 信息。

2. 内置连接池
   Redis 客户端不是每次请求都新建连接，而是复用连接池。

3. 支持单机、哨兵、集群
   可以使用普通 Client、FailoverClient、ClusterClient。

4. 支持 Pipeline、TxPipeline、Lua 脚本
   可以减少网络往返，保证部分复杂操作的原子性。

5. API 风格贴近 Redis 命令
   比如 `Get`、`Set`、`HGet`、`ZAdd`、`Eval` 等。

工程里初始化 Redis 客户端时，重点关注：

* Addr
* Password
* DB
* PoolSize
* MinIdleConns
* DialTimeout
* ReadTimeout
* WriteTimeout
* PoolTimeout
* MaxRetries

面试里不要只说“我用 go-redis 连 Redis”，要补一句：

> 在生产环境中，我会给 Redis 调用统一设置 context timeout，并合理配置连接池和读写超时，避免 Redis 抖动时把 Go 服务 goroutine 卡死。

---

## 【代码块 1】go-redis 基本初始化

```go
package main

import (
	"context"
	"fmt"
	"time"

	"github.com/redis/go-redis/v9"
)

func main() {
	ctx, cancel := context.WithTimeout(context.Background(), 2*time.Second)
	defer cancel()

	rdb := redis.NewClient(&redis.Options{
		Addr:         "localhost:6379",
		Password:     "",
		DB:           0,
		PoolSize:     20,
		MinIdleConns: 5,
		DialTimeout:  2 * time.Second,
		ReadTimeout:  1 * time.Second,
		WriteTimeout: 1 * time.Second,
		PoolTimeout:  2 * time.Second,
		MaxRetries:   2,
	})

	if err := rdb.Ping(ctx).Err(); err != nil {
		fmt.Println("redis ping failed:", err)
		return
	}

	fmt.Println("redis connected")
}
```

---

## 【文本块 3】代码解释

这里有几个重点：

第一，Redis 操作带了 context timeout。
如果 Redis 卡住或网络不通，调用不会无限等待。

第二，客户端配置了连接池。
`PoolSize` 控制最大连接数，`MinIdleConns` 控制最小空闲连接数。

第三，设置了 DialTimeout、ReadTimeout、WriteTimeout。
这是生产环境必须关注的参数，否则下游 Redis 一旦异常，调用方可能被拖垮。

第四，`redis.NewClient` 创建出来的是并发安全的客户端。
通常整个服务共用一个 Redis client，而不是每个请求创建一个。

---

## 【文本块 4】追问：go-redis 客户端是线程安全吗？需要每次请求 new 一个吗？

go-redis 的 Client 是并发安全的。

在 Go Web 服务中，通常会在服务启动时初始化一个全局 Redis client，然后注入到 service 或 repository 里复用。

不要每次请求都 new 一个 client，因为这样会导致连接池失效，频繁建立 TCP 连接，性能很差。

面试可以这么回答：

go-redis 的 Client 内部维护连接池，是 goroutine-safe 的。生产环境一般在应用启动时初始化一次，作为依赖注入到各业务模块中复用，而不是每个请求创建一个客户端。

---

# 二、Redis 数据类型

## 【文本块 5】Q：Redis 常用数据类型有哪些？Go 项目中怎么用？

Redis 常用的基础数据类型有五种：

1. String
2. Hash
3. List
4. Set
5. ZSet

此外还有一些特殊数据类型：

1. Bitmap
2. HyperLogLog
3. GEO
4. Stream

在 Go 项目中，我一般这样使用：

String：
最常用，适合做对象缓存、计数器、分布式锁、验证码、token 黑名单等。

Hash：
适合存储对象的多个字段，比如用户信息、购物车、配置项。优点是可以单独更新某个 field。

List：
适合简单队列、最近记录、时间线。但如果是可靠消息队列，生产上更推荐 Kafka/RocketMQ/RabbitMQ 或 Redis Stream。

Set：
适合去重、标签、共同好友、抽奖、黑名单。

ZSet：
适合排行榜、延时队列、按权重排序、按时间线排序。

Bitmap：
适合签到、在线状态、用户是否访问过等二值统计。

HyperLogLog：
适合大规模 UV 统计，允许少量误差，内存占用极低。

Stream：
适合 Redis 内部的轻量消息流，有消费组、ack、pending 等能力，但如果是核心业务消息，还是要评估专业 MQ。

---

# 三、String 与 SDS

## 【文本块 6】Q：Redis String 底层是怎么实现的？

Redis String 底层不是直接使用 C 语言原生字符串，而是使用 SDS，也就是 Simple Dynamic String。

SDS 相比 C 字符串有几个优势：

第一，获取长度是 O(1)。
C 字符串要通过 `\0` 遍历到末尾才能知道长度，而 SDS 内部保存了 len 字段。

第二，二进制安全。
C 字符串遇到 `\0` 就认为结束，而 SDS 依赖 len 判断长度，所以可以存储图片、音频、序列化后的 protobuf、压缩数据等二进制内容。

第三，避免缓冲区溢出。
SDS 在拼接时会检查容量，不够会自动扩容，不像 C 字符串容易写越界。

第四，减少内存重分配。
SDS 有空间预分配和惰性释放机制，扩容时可能多分配一些空间，缩容时不一定立刻释放，减少频繁 realloc。

Redis String 最大可以存储 512MB，但工程上绝对不建议存这么大的 value。一般超过几十 KB 就要警惕 BigKey 问题。

---

## 【文本块 7】Go 项目里 String 怎么用？

Go 项目里最常见的是把结构体序列化成 JSON 存到 Redis String。

优点：

* 简单
* 通用
* 方便跨语言
* 适合缓存完整对象

缺点：

* 更新某个字段时，需要整体反序列化和重新写入
* JSON 体积较大
* 对热点大对象不友好

如果数据结构很简单，且需要频繁更新单个字段，可以考虑 Hash。
如果对象整体读写为主，可以用 String 存 JSON。
如果追求更小体积和更快速度，可以考虑 protobuf、msgpack 等二进制序列化。

---

## 【代码块 2】String 缓存 JSON 对象

```go
package main

import (
	"context"
	"encoding/json"
	"fmt"
	"time"

	"github.com/redis/go-redis/v9"
)

type User struct {
	ID   int64  `json:"id"`
	Name string `json:"name"`
	Age  int    `json:"age"`
}

func CacheUser(ctx context.Context, rdb *redis.Client, user User) error {
	key := fmt.Sprintf("user:%d", user.ID)

	data, err := json.Marshal(user)
	if err != nil {
		return fmt.Errorf("marshal user: %w", err)
	}

	if err := rdb.Set(ctx, key, data, 10*time.Minute).Err(); err != nil {
		return fmt.Errorf("redis set user cache: %w", err)
	}

	return nil
}

func GetCachedUser(ctx context.Context, rdb *redis.Client, id int64) (*User, error) {
	key := fmt.Sprintf("user:%d", id)

	data, err := rdb.Get(ctx, key).Bytes()
	if err == redis.Nil {
		return nil, nil
	}
	if err != nil {
		return nil, fmt.Errorf("redis get user cache: %w", err)
	}

	var user User
	if err := json.Unmarshal(data, &user); err != nil {
		return nil, fmt.Errorf("unmarshal user cache: %w", err)
	}

	return &user, nil
}
```

---

## 【文本块 8】工程注意点

这里有几个 Go 项目里的 Redis 缓存细节：

第一，区分 `redis.Nil` 和真实错误。
`redis.Nil` 表示 key 不存在，不应该当成系统错误。

第二，缓存反序列化失败要考虑删除脏缓存。
如果 JSON 格式不对，说明缓存可能被污染，可以记录日志并删除该 key。

第三，必须设置 TTL。
缓存数据一般不要永久存在，否则容易产生脏数据和内存膨胀。

第四，key 要规范。
比如：

```text
user:{id}
order:{id}
product:{id}
rank:daily:{date}
```

不要随便拼接没有规范的 key，否则后期排查和治理 BigKey、HotKey 会很痛苦。

---

# 四、Hash

## 【文本块 9】Q：Redis Hash 适合什么场景？

Hash 适合存储对象的多个字段。

比如用户信息：

```text
HSET user:1001 name Tom age 18 city Beijing
```

购物车：

```text
HSET cart:1001 product:1 2 product:2 5
```

Hash 的优势是可以单独读写某个字段，不需要像 String JSON 那样整体反序列化、整体更新。

底层实现上，Hash 在数据量小、字段短时通常使用 listpack 这类紧凑结构节省内存；当字段数量或 value 长度超过阈值，会转换为 hashtable，以提升查询和更新效率。

面试回答：

> Redis Hash 适合对象型数据，尤其是需要频繁更新单个字段的场景。小 Hash 会用紧凑结构节省内存，大 Hash 会转为哈希表提升读写效率。

---

## 【代码块 3】Go 使用 Hash 存用户字段

```go
package main

import (
	"context"
	"fmt"

	"github.com/redis/go-redis/v9"
)

func SaveUserHash(ctx context.Context, rdb *redis.Client, id int64) error {
	key := fmt.Sprintf("user:%d", id)

	err := rdb.HSet(ctx, key, map[string]any{
		"name": "Tom",
		"age":  18,
		"city": "Beijing",
	}).Err()
	if err != nil {
		return fmt.Errorf("hset user: %w", err)
	}

	return nil
}

func GetUserName(ctx context.Context, rdb *redis.Client, id int64) (string, error) {
	key := fmt.Sprintf("user:%d", id)

	name, err := rdb.HGet(ctx, key, "name").Result()
	if err == redis.Nil {
		return "", nil
	}
	if err != nil {
		return "", fmt.Errorf("hget user name: %w", err)
	}

	return name, nil
}
```

---

## 【文本块 10】Hash 和 String JSON 怎么选？

如果对象通常整体读写，比如商品详情页、用户资料页，String JSON 更简单。

如果对象字段很多，但业务经常只读写其中几个字段，Hash 更合适。

如果字段数量非常多，比如一个 Hash 里放几十万 field，就要警惕 BigKey。此时应该拆分，比如：

```text
user:profile:{id}
user:stat:{id}
cart:{userID}:{bucket}
```

面试可以这样回答：

String JSON 简单，适合整体缓存；Hash 支持字段级读写，适合部分更新。但 Hash 不能无限大，大 Hash 会带来 BigKey 问题，需要拆分。

---

# 五、List

## 【文本块 11】Q：Redis List 底层是什么？适合什么场景？

Redis List 是一个有序列表，可以从两端 push/pop。

常见命令：

```text
LPUSH
RPUSH
LPOP
RPOP
BLPOP
BRPOP
LRANGE
```

底层在 Redis 新版本中主要是 quicklist/listpack 思路：既保留链表方便两端操作，又用紧凑结构减少内存碎片。

List 适合：

* 最近 N 条记录
* 简单队列
* 时间线
* 阻塞消费

但是如果面试问“能不能用 List 做消息队列”，要回答得谨慎。

List 可以做简单队列，比如 `LPUSH + BRPOP`，但它缺少完善的 ack、重试、消费组、消息堆积管理等能力。核心业务消息更推荐专业 MQ 或 Redis Stream。

---

## 【代码块 4】Go 使用 List 做简单队列

```go
package main

import (
	"context"
	"fmt"
	"time"

	"github.com/redis/go-redis/v9"
)

func PushTask(ctx context.Context, rdb *redis.Client, task string) error {
	if err := rdb.LPush(ctx, "task:queue", task).Err(); err != nil {
		return fmt.Errorf("lpush task: %w", err)
	}
	return nil
}

func PopTask(ctx context.Context, rdb *redis.Client) (string, error) {
	result, err := rdb.BRPop(ctx, 5*time.Second, "task:queue").Result()
	if err == redis.Nil {
		return "", nil
	}
	if err != nil {
		return "", fmt.Errorf("brpop task: %w", err)
	}

	if len(result) != 2 {
		return "", fmt.Errorf("unexpected brpop result: %v", result)
	}

	return result[1], nil
}
```

---

## 【文本块 12】List 做队列有什么问题？

主要问题有三个：

第一，可靠性不足。
消费者取走消息后，如果处理失败，消息已经从 List 删除了。

第二，不方便做消费组。
多个消费者的分组、offset、pending 管理都要自己实现。

第三，不方便做重试和死信。
失败消息如何重试、重试多少次、失败后进入哪里，都不是 List 原生强项。

所以 List 更适合轻量场景，不适合核心业务消息系统。

---

# 六、Set

## 【文本块 13】Q：Redis Set 适合什么场景？

Set 是无序去重集合。

常见用途：

* 用户标签
* 黑名单
* 抽奖去重
* 点赞用户集合
* 共同好友
* 交集、并集、差集

常用命令：

```text
SADD
SREM
SISMEMBER
SMEMBERS
SCARD
SINTER
SUNION
SDIFF
```

底层实现上，如果元素都是整数且数量较小，Redis 可能用 intset 节省内存；否则使用 hashtable。

面试可以这么回答：

Set 的核心价值是去重和集合运算。比如共同好友可以用交集，用户标签可以用集合，抽奖可以用 Set 保证一个用户只参与一次。

---

## 【代码块 5】Go 使用 Set 做标签

```go
package main

import (
	"context"
	"fmt"

	"github.com/redis/go-redis/v9"
)

func AddUserTags(ctx context.Context, rdb *redis.Client, userID int64, tags ...string) error {
	key := fmt.Sprintf("user:tags:%d", userID)

	values := make([]any, 0, len(tags))
	for _, tag := range tags {
		values = append(values, tag)
	}

	if err := rdb.SAdd(ctx, key, values...).Err(); err != nil {
		return fmt.Errorf("sadd user tags: %w", err)
	}

	return nil
}

func HasTag(ctx context.Context, rdb *redis.Client, userID int64, tag string) (bool, error) {
	key := fmt.Sprintf("user:tags:%d", userID)

	ok, err := rdb.SIsMember(ctx, key, tag).Result()
	if err != nil {
		return false, fmt.Errorf("sismember user tag: %w", err)
	}

	return ok, nil
}
```

---

# 七、ZSet 与跳表

## 【文本块 14】Q：ZSet 底层是怎么实现的？为什么用跳表？

ZSet 是有序集合，每个 member 对应一个 score，Redis 会按 score 排序。

常见用途：

* 排行榜
* 延时队列
* 热度榜
* 时间线
* 优先级队列

底层实现：

当元素少、member 短时，使用 listpack 紧凑存储，节省内存。

当数据量变大后，使用 skiplist + hashtable。

hashtable 用于通过 member 快速查 score。
skiplist 用于按 score 排序、范围查询、排名查询。

为什么用跳表而不是红黑树？

第一，实现简单。
跳表比红黑树更容易实现和维护。

第二，范围查询方便。
ZSet 经常做 `ZRANGE`、`ZRANGEBYSCORE`。跳表找到起点后，顺着底层链表往后遍历即可。

第三，性能稳定。
跳表查询、插入、删除平均复杂度都是 O(logN)。

面试回答：

> ZSet 大数据量时底层是 skiplist + hashtable。hashtable 保证按 member 查找 O(1)，skiplist 保证按 score 排序和范围查询 O(logN)。Redis 选择跳表主要是因为实现简单、范围查询方便、性能足够稳定。

---

## 【代码块 6】Go 使用 ZSet 做排行榜

```go
package main

import (
	"context"
	"fmt"

	"github.com/redis/go-redis/v9"
)

func AddScore(ctx context.Context, rdb *redis.Client, userID int64, score float64) error {
	key := "rank:daily:2026-06-14"

	member := fmt.Sprintf("%d", userID)

	err := rdb.ZAdd(ctx, key, redis.Z{
		Score:  score,
		Member: member,
	}).Err()
	if err != nil {
		return fmt.Errorf("zadd rank: %w", err)
	}

	return nil
}

func TopN(ctx context.Context, rdb *redis.Client, n int64) ([]redis.Z, error) {
	key := "rank:daily:2026-06-14"

	result, err := rdb.ZRevRangeWithScores(ctx, key, 0, n-1).Result()
	if err != nil {
		return nil, fmt.Errorf("zrevrange rank: %w", err)
	}

	return result, nil
}
```

---

## 【文本块 15】ZSet 做延时队列

ZSet 可以用 score 存任务执行时间戳。

生产者：

```text
ZADD delay:queue executeAt taskID
```

消费者定期扫描：

```text
ZRANGEBYSCORE delay:queue -inf now LIMIT 0 10
```

取到任务后，再用 `ZREM` 删除。

但是这里有并发问题：多个消费者可能同时取到同一个任务。所以删除任务时要保证“取任务 + 删除任务”原子性，通常用 Lua 脚本。

---

## 【代码块 7】ZSet 延时队列 Lua 原子取任务

```go
package main

import (
	"context"
	"fmt"
	"time"

	"github.com/redis/go-redis/v9"
)

var popDueTasksScript = redis.NewScript(`
local key = KEYS[1]
local now = ARGV[1]
local limit = tonumber(ARGV[2])

local tasks = redis.call('ZRANGEBYSCORE', key, '-inf', now, 'LIMIT', 0, limit)
if #tasks == 0 then
    return {}
end

for i, task in ipairs(tasks) do
    redis.call('ZREM', key, task)
end

return tasks
`)

func PopDueTasks(ctx context.Context, rdb *redis.Client, key string, limit int) ([]string, error) {
	now := time.Now().UnixMilli()

	result, err := popDueTasksScript.Run(ctx, rdb, []string{key}, now, limit).StringSlice()
	if err != nil {
		return nil, fmt.Errorf("pop due tasks: %w", err)
	}

	return result, nil
}
```

---

## 【文本块 16】延时队列工程注意点

ZSet 延时队列适合轻量任务，不适合强可靠核心消息。

需要注意：

1. 消费失败如何重试？
2. 任务处理超时怎么办？
3. 消费者宕机后任务是否丢失？
4. 是否需要 ack？
5. 是否需要死信队列？
6. 是否需要幂等？

如果这些要求都很强，建议使用专业 MQ 的延迟消息能力。

---

# 八、Bitmap 与 HyperLogLog

## 【文本块 17】Q：如果统计 1 亿用户签到，怎么设计？

签到只有两种状态：签了或没签。

这种二值状态非常适合 Bitmap。

可以用日期作为 key，用户 ID 作为 offset：

```text
SETBIT sign:2026-06-14 userID 1
GETBIT sign:2026-06-14 userID
BITCOUNT sign:2026-06-14
```

优点是内存极省。

1 亿用户需要 1 亿 bit，大约 12MB 左右，比用 Set 存 1 亿个用户 ID 小很多。

但是 Bitmap 的问题是 offset 不能太稀疏。
如果用户 ID 很大，比如从 10 亿开始，直接用 userID 做 offset 会浪费大量空间。此时要做 ID 映射或分片。

---

## 【代码块 8】Go 使用 Bitmap 做签到

```go
package main

import (
	"context"
	"fmt"

	"github.com/redis/go-redis/v9"
)

func Sign(ctx context.Context, rdb *redis.Client, date string, userID int64) error {
	key := fmt.Sprintf("sign:%s", date)

	if err := rdb.SetBit(ctx, key, userID, 1).Err(); err != nil {
		return fmt.Errorf("setbit sign: %w", err)
	}

	return nil
}

func IsSigned(ctx context.Context, rdb *redis.Client, date string, userID int64) (bool, error) {
	key := fmt.Sprintf("sign:%s", date)

	v, err := rdb.GetBit(ctx, key, userID).Result()
	if err != nil {
		return false, fmt.Errorf("getbit sign: %w", err)
	}

	return v == 1, nil
}
```

---

## 【文本块 18】Q：如果统计日活 UV，怎么设计？

如果要求精确统计，可以用 Set，把访问过的 userID 加进去：

```text
SADD uv:2026-06-14 userID
SCARD uv:2026-06-14
```

但如果用户量很大，Set 会非常占内存。

如果允许少量误差，可以用 HyperLogLog：

```text
PFADD uv:2026-06-14 userID
PFCOUNT uv:2026-06-14
```

HyperLogLog 的优点是内存固定且很小，Redis 中每个 HLL 大约 12KB，适合大规模 UV 统计。

缺点是：

1. 有误差
2. 只能统计基数
3. 不能取出具体 userID

面试回答：

> 精确去重用 Set，海量 UV 近似统计用 HyperLogLog。HyperLogLog 内存极小，但不能反查具体用户，也有少量误差。

---

# 九、Redis 高可用

## 【文本块 19】Q：Redis 是内存数据库，宕机了数据怎么办？

Redis 主要有两种持久化机制：

第一，RDB。
定期生成内存快照，保存到磁盘。

优点：

* 文件紧凑
* 恢复快
* 适合备份

缺点：

* 两次快照之间的数据可能丢失
* fork 生成 RDB 时可能有额外开销

第二，AOF。
把写命令追加到日志文件中。

优点：

* 数据更安全
* 可以配置每秒 fsync，最多丢 1 秒数据

缺点：

* 文件更大
* 恢复通常比 RDB 慢
* 需要 rewrite 压缩

生产里常见的是混合持久化。
RDB 负责快速恢复基线数据，AOF 记录增量写命令，兼顾恢复速度和数据安全。

面试可以这样回答：

Redis 持久化主要有 RDB 和 AOF。RDB 是快照，恢复快但可能丢快照间隔内的数据；AOF 是追加写日志，数据更安全但文件更大。生产里通常用混合持久化，在恢复速度和数据安全之间折中。

---

## 【文本块 20】Q：Redis 主从复制有什么用？

主从复制主要解决两个问题：

第一，读扩展。
主节点负责写，从节点负责读，可以分摊读压力。

第二，高可用基础。
主节点挂了以后，从节点可以被提升为主节点。

但要注意：

主从复制本身不等于自动故障转移。
如果没有哨兵或集群，主挂了以后还需要人工切换。

主从复制通常是异步的，因此有主从延迟，甚至在故障切换时可能丢数据。

---

## 【文本块 21】Q：主从同步是怎么做的？

主从同步主要分为全量同步和增量同步。

第一次建立复制关系时，通常会全量同步：

1. 从节点向主节点发起同步请求
2. 主节点生成 RDB
3. 主节点把 RDB 发给从节点
4. 从节点加载 RDB
5. 主节点继续把同步期间产生的新写命令发给从节点

如果连接短暂断开，且复制积压缓冲区里还保留了断开期间的命令，就可以做增量同步。

如果断开太久，积压缓冲区不够，就只能重新全量同步。

优化思路：

* 合理配置复制积压缓冲区
* 避免大 key、大事务导致同步压力
* 控制从节点数量
* 核心读请求需要考虑主从延迟

---

## 【文本块 22】Q：哨兵 Sentinel 是做什么的？

Sentinel 是 Redis 主从架构下的自动故障转移机制。

它主要负责：

1. 监控主从节点是否存活
2. 判断主节点是否下线
3. 多个哨兵投票确认客观下线
4. 从从节点中选出新主节点
5. 通知客户端新的主节点地址

哨兵解决的是“主节点挂了以后自动切换”的问题。

但哨兵不解决：

* 单机内存瓶颈
* 单主写压力瓶颈
* 数据分片问题

所以如果需要水平扩容，要用 Redis Cluster。

---

## 【代码块 9】go-redis 连接 Sentinel

```go
package main

import (
	"context"
	"fmt"
	"time"

	"github.com/redis/go-redis/v9"
)

func main() {
	ctx, cancel := context.WithTimeout(context.Background(), 2*time.Second)
	defer cancel()

	rdb := redis.NewFailoverClient(&redis.FailoverOptions{
		MasterName:    "mymaster",
		SentinelAddrs: []string{"127.0.0.1:26379", "127.0.0.1:26380"},
		Password:      "",
		DB:            0,
		PoolSize:      20,
	})

	if err := rdb.Ping(ctx).Err(); err != nil {
		fmt.Println("redis sentinel ping failed:", err)
		return
	}

	fmt.Println("redis sentinel connected")
}
```

---

## 【文本块 23】Q：Redis Cluster 是什么？

Redis Cluster 用于解决单机容量和单主写入瓶颈。

它把整个 key 空间划分为 16384 个哈希槽。
每个 key 根据 CRC16 计算出槽位：

```text
slot = CRC16(key) % 16384
```

不同槽位分布在不同主节点上。

客户端访问 key 时，先计算 slot，再找到对应节点。

Redis Cluster 的优势：

* 支持水平扩容
* 多主分片
* 节点故障可自动转移
* 容量和写入能力可以横向扩展

代价：

* 运维复杂度增加
* 跨槽多 key 操作受限
* pipeline、事务、Lua 跨槽受限
* 客户端要支持 MOVED/ASK 重定向

---

## 【代码块 10】go-redis 连接 Cluster

```go
package main

import (
	"context"
	"fmt"
	"time"

	"github.com/redis/go-redis/v9"
)

func main() {
	ctx, cancel := context.WithTimeout(context.Background(), 2*time.Second)
	defer cancel()

	rdb := redis.NewClusterClient(&redis.ClusterOptions{
		Addrs: []string{
			"127.0.0.1:7000",
			"127.0.0.1:7001",
			"127.0.0.1:7002",
		},
		PoolSize:    30,
		ReadTimeout: time.Second,
	})

	if err := rdb.Ping(ctx).Err(); err != nil {
		fmt.Println("redis cluster ping failed:", err)
		return
	}

	fmt.Println("redis cluster connected")
}
```

---

## 【文本块 24】Redis 高可用怎么整体回答？

我会按四层讲：

第一，持久化。
RDB/AOF 解决宕机重启后的数据恢复问题。

第二，主从复制。
解决读扩展和数据副本问题。

第三，哨兵。
解决主节点故障后的自动选主和切换问题。

第四，Cluster。
解决单机容量和单主写瓶颈问题。

一句话：

> Redis 高可用可以按“恢复、复制、切换、扩容”四个层次来讲：持久化负责恢复，主从负责复制，哨兵负责故障切换，Cluster 负责分片扩容。

---

# 十、过期策略与内存淘汰

## 【文本块 25】Q：Redis 的 key 过期了会立刻删除吗？

不会。

Redis key 过期后，并不是立刻删除，而是通过惰性删除 + 定期删除配合处理。

惰性删除：
当客户端访问某个 key 时，Redis 会检查它是否过期。如果过期，就删除并返回空。

优点是节省 CPU。
缺点是如果过期 key 一直没人访问，它会继续占内存。

定期删除：
Redis 会周期性随机抽取一批设置了过期时间的 key，检查是否过期，过期则删除。

为什么不全量扫描？

因为 Redis 是高性能服务，如果每次全量扫描所有 key，会造成明显阻塞。

所以 Redis 在 CPU 和内存之间做了折中：

惰性删除保证访问时清理。
定期删除负责兜底回收。
内存不够时再走内存淘汰策略。

---

## 【文本块 26】Q：内存满了 Redis 会怎么做？

当 Redis 使用内存达到 maxmemory 限制时，会触发内存淘汰策略。

常见策略：

1. noeviction
   默认策略，不淘汰，写入直接报错。

2. allkeys-lru
   从所有 key 中淘汰最近最少使用的数据。

3. volatile-lru
   只从设置了过期时间的 key 中淘汰最近最少使用的数据。

4. allkeys-lfu
   从所有 key 中淘汰访问频率最低的数据。

5. volatile-lfu
   只从设置了过期时间的 key 中淘汰访问频率最低的数据。

6. allkeys-random
   从所有 key 中随机淘汰。

7. volatile-random
   从设置了过期时间的 key 中随机淘汰。

8. volatile-ttl
   优先淘汰剩余 TTL 更短的 key。

工程里如果 Redis 主要作为缓存，通常选择：

```text
allkeys-lru
```

或者在存在扫描污染问题时选择：

```text
allkeys-lfu
```

不要生产环境长期使用 noeviction 作为缓存策略，否则内存满了以后写入会失败，影响业务。

---

## 【文本块 27】Q：Redis 的 LRU 是标准 LRU 吗？

不是严格标准 LRU，而是近似 LRU。

标准 LRU 通常需要维护一个全局双向链表，每次访问 key 都要移动节点到链表头。

这样有两个问题：

1. 每个 key 都要额外维护前后指针，增加内存。
2. 高频访问时维护链表成本高。

Redis 采用随机采样。

当需要淘汰时，随机抽取若干 key，比较它们的访问时间，淘汰其中最久未访问的 key。

采样数量越大，越接近标准 LRU，但 CPU 成本也越高。

面试回答：

> Redis 的 LRU 是近似 LRU，通过随机采样降低内存和维护成本。它不是全局维护链表的严格 LRU，但在采样数合适时效果已经接近标准 LRU。

---

## 【文本块 28】LRU 和 LFU 有什么区别？

LRU 关注最近访问时间。
LFU 关注访问频率。

LRU 的问题是容易被偶发扫描污染。

比如 Redis 里原本有很多热点 key，突然有一个任务全量扫描一批冷数据，这些冷数据因为刚被访问过，可能在 LRU 里变得“很新”，从而挤掉真正热点数据。

LFU 更关注历史访问频率，可以缓解这个问题。即使冷数据刚被访问一次，因为频率低，也不会轻易挤掉长期热点 key。

面试可以这样回答：

如果业务热点符合“最近访问过就可能继续访问”，LRU 比较合适；如果存在批量扫描、偶发访问污染缓存的问题，LFU 更稳，因为它关注访问频率，而不只是最近一次访问时间。

---

# 十一、缓存穿透

## 【文本块 29】Q：什么是缓存穿透？怎么解决？

缓存穿透是指查询一个根本不存在的数据。

流程是：

1. 请求查 Redis
2. Redis 没有
3. 请求打到数据库
4. 数据库也没有
5. 不写缓存
6. 下次同样请求继续打数据库

如果有人恶意构造大量不存在的 ID，就会绕过缓存，直接压垮数据库。

解决方案主要有两个：

第一，缓存空对象。
数据库查不到时，也往 Redis 写一个空值，并设置较短 TTL。

优点是简单。
缺点是会缓存很多无效 key，并且存在短暂一致性问题。

第二，布隆过滤器。
在访问 Redis 之前，先判断 key 是否可能存在。如果布隆过滤器说不存在，直接返回，不访问 Redis 和 DB。

优点是内存小、性能高。
缺点是有误判。它说不存在一定不存在；它说存在只是可能存在。

---

## 【代码块 11】Go 缓存空对象防穿透

```go
package main

import (
	"context"
	"encoding/json"
	"fmt"
	"time"

	"github.com/redis/go-redis/v9"
)

type Product struct {
	ID   int64  `json:"id"`
	Name string `json:"name"`
}

func GetProduct(ctx context.Context, rdb *redis.Client, id int64) (*Product, error) {
	key := fmt.Sprintf("product:%d", id)

	data, err := rdb.Get(ctx, key).Bytes()
	if err == nil {
		if len(data) == 0 {
			return nil, nil
		}

		var p Product
		if err := json.Unmarshal(data, &p); err != nil {
			return nil, fmt.Errorf("unmarshal product cache: %w", err)
		}
		return &p, nil
	}

	if err != redis.Nil {
		return nil, fmt.Errorf("redis get product: %w", err)
	}

	product, err := QueryProductFromDB(ctx, id)
	if err != nil {
		return nil, err
	}

	if product == nil {
		_ = rdb.Set(ctx, key, "", 2*time.Minute).Err()
		return nil, nil
	}

	b, err := json.Marshal(product)
	if err != nil {
		return nil, fmt.Errorf("marshal product: %w", err)
	}

	_ = rdb.Set(ctx, key, b, 10*time.Minute).Err()

	return product, nil
}

func QueryProductFromDB(ctx context.Context, id int64) (*Product, error) {
	return nil, nil
}
```

---

## 【文本块 30】缓存空对象注意点

缓存空对象要设置短 TTL，比如 1 到 5 分钟。

否则如果后面数据库真的插入了这个 ID，对应缓存仍然是空，会导致短时间内查不到新数据。

另外，如果恶意请求 ID 非常随机，缓存空对象会产生大量垃圾 key，此时布隆过滤器更合适。

---

# 十二、缓存击穿

## 【文本块 31】Q：什么是缓存击穿？怎么解决？

缓存击穿指的是某一个热点 key 突然失效，导致大量并发请求同时打到数据库。

它和缓存穿透不同。

穿透是查不存在的数据。
击穿是查存在的热点数据，但热点 key 过期了。

解决方案主要有三类：

第一，互斥锁。
缓存失效后，只有一个请求能去查数据库并重建缓存，其他请求等待或重试。

第二，逻辑过期。
Redis key 不设置物理过期时间，而是在 value 里存一个逻辑过期时间。请求发现逻辑过期后，先返回旧数据，同时异步重建缓存。

第三，singleflight。
在 Go 单进程内，可以用 `singleflight` 合并同一 key 的并发请求，让同一时刻只有一个 goroutine 去查 DB。

面试里 Go 版一定要提 singleflight，这是 Go 项目里防击穿非常常用的工具。

---

## 【代码块 12】Go 使用 singleflight 防缓存击穿

```go
package main

import (
	"context"
	"encoding/json"
	"fmt"
	"time"

	"github.com/redis/go-redis/v9"
	"golang.org/x/sync/singleflight"
)

type User struct {
	ID   int64  `json:"id"`
	Name string `json:"name"`
}

var userGroup singleflight.Group

func GetUserWithSingleflight(ctx context.Context, rdb *redis.Client, id int64) (*User, error) {
	key := fmt.Sprintf("user:%d", id)

	data, err := rdb.Get(ctx, key).Bytes()
	if err == nil {
		var user User
		if err := json.Unmarshal(data, &user); err != nil {
			return nil, fmt.Errorf("unmarshal user cache: %w", err)
		}
		return &user, nil
	}
	if err != redis.Nil {
		return nil, fmt.Errorf("redis get user: %w", err)
	}

	v, err, _ := userGroup.Do(key, func() (any, error) {
		// 双重检查，防止等待期间缓存已经被其他请求重建
		data, err := rdb.Get(ctx, key).Bytes()
		if err == nil {
			var user User
			if err := json.Unmarshal(data, &user); err != nil {
				return nil, fmt.Errorf("unmarshal user cache after singleflight: %w", err)
			}
			return &user, nil
		}
		if err != redis.Nil {
			return nil, fmt.Errorf("redis get user after singleflight: %w", err)
		}

		user, err := QueryUserFromDB(ctx, id)
		if err != nil {
			return nil, err
		}
		if user == nil {
			return nil, nil
		}

		b, err := json.Marshal(user)
		if err != nil {
			return nil, fmt.Errorf("marshal user: %w", err)
		}

		_ = rdb.Set(ctx, key, b, 10*time.Minute).Err()
		return user, nil
	})
	if err != nil {
		return nil, err
	}
	if v == nil {
		return nil, nil
	}

	return v.(*User), nil
}

func QueryUserFromDB(ctx context.Context, id int64) (*User, error) {
	return &User{ID: id, Name: "Tom"}, nil
}
```

---

## 【文本块 32】singleflight 的局限性

singleflight 只能合并当前 Go 进程内的并发请求。

如果服务部署了 100 个实例，每个实例都会有一个请求打到数据库，所以它不能完全替代分布式锁。

但在很多场景里，singleflight 已经能显著降低压力，而且没有分布式锁那么复杂。

更强的方案可以组合：

* 本地 singleflight
* Redis 分布式互斥锁
* 热点 key 永不过期 + 逻辑过期
* 后台主动刷新缓存

---

## 【文本块 33】互斥锁和逻辑过期怎么选？

互斥锁：

优点：

* 数据更准确
* 缓存重建逻辑简单

缺点：

* 请求会等待
* 锁竞争时延迟上升
* 要处理锁超时和死锁

适合：

* 商品详情
* 用户信息
* 准确性要求更高的场景

逻辑过期：

优点：

* 性能好
* 请求不会阻塞
* 可用性高

缺点：

* 会短暂返回旧数据
* 实现复杂，需要异步刷新

适合：

* 热榜
* 推荐内容
* 文章详情
* 对短暂旧数据可接受的场景

面试可以这样回答：

对一致性要求较高的热点 key，用互斥锁重建；对可用性和性能要求更高、允许短暂旧数据的热点 key，用逻辑过期异步刷新。

---

# 十三、缓存雪崩

## 【文本块 34】Q：什么是缓存雪崩？怎么解决？

缓存雪崩是指大量缓存 key 在同一时间失效，或者 Redis 整体不可用，导致请求大面积打到数据库。

常见原因：

1. 大量 key 设置了相同 TTL
2. Redis 节点宕机
3. Redis 集群故障
4. 缓存预热不足
5. 热点数据集中失效

解决方案：

第一，随机 TTL。
设置缓存过期时间时加随机抖动，避免同一时刻集中失效。

第二，缓存预热。
活动开始前提前加载热点数据到 Redis。

第三，高可用架构。
使用主从、哨兵、Cluster，避免单点故障。

第四，本地缓存兜底。
比如 Ristretto、BigCache、FreeCache，Redis 短暂抖动时可以读本地旧数据。

第五，限流、熔断、降级。
Redis 或 DB 异常时，不让所有请求继续打到底层。

第六，多级缓存。
本地缓存 + Redis + DB，逐层兜底。

---

## 【代码块 13】设置随机 TTL

```go
package main

import (
	"context"
	"math/rand"
	"time"

	"github.com/redis/go-redis/v9"
)

func SetCacheWithJitter(ctx context.Context, rdb *redis.Client, key string, value any, baseTTL time.Duration) error {
	jitter := time.Duration(rand.Intn(300)) * time.Second
	ttl := baseTTL + jitter

	return rdb.Set(ctx, key, value, ttl).Err()
}
```

---

## 【文本块 35】缓存雪崩回答模板

缓存雪崩我会分两类看：

如果是大量 key 同时过期，就通过随机 TTL、缓存预热、热点 key 逻辑过期来解决。

如果是 Redis 服务不可用，就通过 Redis 高可用、本地缓存、限流熔断、服务降级来保护数据库。

一句话：

> 雪崩的核心是避免“同一时间大量请求绕过缓存打到底层”，所以要从过期时间打散、缓存预热、高可用架构和降级限流四个层面处理。

---

# 十四、缓存与数据库双写一致性

## 【文本块 36】Q：数据库更新后，是更新缓存还是删除缓存？

一般推荐：先更新数据库，再删除缓存。

不是更新缓存，而是删除缓存。

为什么删除而不是更新？

第一，更新缓存容易浪费。
如果写多读少，数据库更新很多次，但缓存可能根本没人读，每次都更新缓存就是浪费。

第二，并发更新容易乱序。
两个线程先后更新 DB，但更新缓存的顺序可能反过来，导致缓存留下旧值。

第三，删除更简单。
删除后，下次读请求发现缓存不存在，再从数据库加载最新数据并回写缓存。

为什么是先更新数据库，再删除缓存？

如果先删缓存，再更新数据库，可能出现：

1. 写请求先删除缓存
2. 读请求发现缓存为空
3. 读请求查数据库，此时还是旧值
4. 读请求把旧值写回缓存
5. 写请求更新数据库为新值

这样缓存会长时间保存旧数据。

所以更推荐：

1. 更新数据库
2. 删除缓存

虽然这个方案也不是绝对完美，因为“数据库更新成功，但删除缓存失败”仍然会不一致。

解决方式是加补偿机制，比如 MQ 重试、binlog 监听、延时双删。

---

## 【代码块 14】先更新 DB 再删除缓存

```go
package main

import (
	"context"
	"fmt"

	"github.com/redis/go-redis/v9"
)

func UpdateUserName(ctx context.Context, rdb *redis.Client, userID int64, name string) error {
	if err := UpdateUserNameInDB(ctx, userID, name); err != nil {
		return fmt.Errorf("update user db: %w", err)
	}

	key := fmt.Sprintf("user:%d", userID)
	if err := rdb.Del(ctx, key).Err(); err != nil {
		// 生产里不能只打印完就算了，应该进入重试队列或发 MQ 补偿
		return fmt.Errorf("delete user cache: %w", err)
	}

	return nil
}

func UpdateUserNameInDB(ctx context.Context, userID int64, name string) error {
	return nil
}
```

---

## 【文本块 37】Q：如果删除缓存失败怎么办？

如果数据库更新成功，但删除缓存失败，就会导致 Redis 里仍然是旧值。

解决方案：

第一，重试。
删除失败后重试几次，但不要无限阻塞主流程。

第二，异步补偿。
把删除缓存任务写入 MQ，由消费者重试删除。

第三，监听 binlog。
通过 Canal、Debezium 等监听 MySQL binlog，捕获数据变更后删除缓存。

第四，设置合理 TTL。
即使删除失败，缓存也会在 TTL 后自然过期，这是最后兜底。

工业级更推荐：

数据库更新成功后提交事务，然后发送可靠消息或通过 outbox/binlog 机制触发缓存删除。

---

## 【文本块 38】延时双删是什么？

延时双删通常流程是：

1. 先删除缓存
2. 更新数据库
3. sleep 一小段时间
4. 再删除一次缓存

它的目的是清理并发读请求可能写回的旧缓存。

但延时双删的问题是：

* sleep 时间不好确定
* 代码侵入业务
* 不能保证一定成功
* 删除失败仍然需要补偿

所以延时双删可以作为简单方案，但更可靠的是 MQ/binlog 补偿删除。

面试可以这样回答：

延时双删能缓解并发读写导致的旧缓存回写问题，但不是强一致方案。生产中如果对一致性要求更高，我更倾向于先更新 DB 再删缓存，并通过 MQ 或 binlog 监听做删除失败补偿。

---

# 十五、分布式锁与 SET NX EX

## 【文本块 39】Q：如何用 Redis 实现分布式锁？

Redis 分布式锁最基本的命令是：

```text
SET lock_key unique_value NX EX 10
```

含义：

* NX：key 不存在才设置
* EX 10：设置 10 秒过期时间
* unique_value：锁持有者唯一标识

为什么不能用 SETNX 后再 EXPIRE？

因为 SETNX 和 EXPIRE 是两条命令，不是原子操作。

如果 SETNX 成功后，服务还没来得及 EXPIRE 就宕机，锁就会永久存在，造成死锁。

所以必须用 `SET key value NX EX seconds` 这种原子命令。

---

## 【代码块 15】Go 实现基础 Redis 锁

```go
package main

import (
	"context"
	"crypto/rand"
	"encoding/hex"
	"fmt"
	"time"

	"github.com/redis/go-redis/v9"
)

type RedisLock struct {
	rdb   *redis.Client
	key   string
	value string
	ttl   time.Duration
}

func NewRedisLock(rdb *redis.Client, key string, ttl time.Duration) (*RedisLock, error) {
	value, err := randomValue()
	if err != nil {
		return nil, err
	}

	return &RedisLock{
		rdb:   rdb,
		key:   key,
		value: value,
		ttl:   ttl,
	}, nil
}

func (l *RedisLock) Lock(ctx context.Context) (bool, error) {
	ok, err := l.rdb.SetNX(ctx, l.key, l.value, l.ttl).Result()
	if err != nil {
		return false, fmt.Errorf("redis setnx lock: %w", err)
	}
	return ok, nil
}

func randomValue() (string, error) {
	b := make([]byte, 16)
	if _, err := rand.Read(b); err != nil {
		return "", fmt.Errorf("rand lock value: %w", err)
	}
	return hex.EncodeToString(b), nil
}
```

---

## 【文本块 40】为什么锁 value 必须是唯一值？

如果 value 只是固定字符串，比如 `"1"`，会出现误删别人的锁。

场景：

1. A 拿到锁，TTL 是 10 秒
2. A 执行业务太慢，锁自动过期
3. B 拿到锁
4. A 业务执行完，执行 DEL
5. A 把 B 的锁删掉了

所以 value 必须是唯一标识，比如：

* UUID
* random token
* instanceID + goroutine/taskID
* requestID

释放锁时，要先判断 value 是不是自己的，是自己的才能删。

---

## 【文本块 41】Q：判断 value 和删除 key 怎么保证原子性？

不能用 Go 代码这样写：

```go
if redis.Get(key) == value {
    redis.Del(key)
}
```

因为 GET 和 DEL 是两步，中间锁可能已经过期并被别人拿到。

正确方式是 Lua 脚本：

```lua
if redis.call("get", KEYS[1]) == ARGV[1] then
    return redis.call("del", KEYS[1])
else
    return 0
end
```

Redis 执行 Lua 脚本时是原子的，中间不会插入其他命令。

---

## 【代码块 16】Lua 原子解锁

```go
package main

import (
	"context"
	"fmt"

	"github.com/redis/go-redis/v9"
)

var unlockScript = redis.NewScript(`
if redis.call("get", KEYS[1]) == ARGV[1] then
    return redis.call("del", KEYS[1])
else
    return 0
end
`)

func (l *RedisLock) Unlock(ctx context.Context) error {
	result, err := unlockScript.Run(ctx, l.rdb, []string{l.key}, l.value).Int64()
	if err != nil {
		return fmt.Errorf("redis unlock script: %w", err)
	}

	if result == 0 {
		return fmt.Errorf("lock not owned or already expired")
	}

	return nil
}
```

---

## 【文本块 42】Redis 分布式锁还有什么问题？

手写 Redis 锁还有几个问题：

第一，不可重入。
同一个业务线程重复加锁会失败，除非自己维护重入计数。

第二，锁续期问题。
如果业务执行时间超过 TTL，锁会提前过期。

第三，主从切换一致性问题。
Redis 主从复制是异步的，主节点加锁成功但还没同步给从节点时主挂了，新主可能没有这把锁，其他客户端又能加锁。

第四，锁不是业务幂等的替代品。
即使有分布式锁，业务本身仍然要做幂等设计。

面试回答要稳：

> Redis 分布式锁适合一般互斥场景，但不能把它当成强一致锁。如果是金融扣款、库存强一致这类场景，不能只依赖 Redis 锁，还要结合数据库事务、唯一约束、状态机和幂等机制。

---

# 十六、看门狗与锁续期

## 【文本块 43】Q：业务执行超过锁过期时间怎么办？

如果锁 TTL 是 10 秒，但业务执行了 30 秒，那么锁会在业务还没完成时过期。其他请求会拿到锁，导致并发执行。

解决办法是锁续期，也叫看门狗机制。

Java 里 Redisson 有 watchdog，默认锁 30 秒，每隔 10 秒自动续期。

Go 里没有一个统一事实标准的 Redisson 等价物。如果使用手写锁，需要自己启动续期 goroutine；也可以使用成熟库，比如 redsync，但仍要理解其语义和限制。

续期的核心逻辑是：

1. 加锁成功
2. 启动后台 goroutine
3. 定期检查锁 value 是否仍然是自己的
4. 如果是自己的，就延长 TTL
5. 解锁或 context 取消时停止续期

续期也必须用 Lua 保证“判断 value + expire”原子性。

---

## 【代码块 17】Lua 原子续期

```go
package main

import (
	"context"
	"fmt"
	"time"

	"github.com/redis/go-redis/v9"
)

var renewScript = redis.NewScript(`
if redis.call("get", KEYS[1]) == ARGV[1] then
    return redis.call("pexpire", KEYS[1], ARGV[2])
else
    return 0
end
`)

func (l *RedisLock) Renew(ctx context.Context) error {
	ttlMillis := l.ttl.Milliseconds()

	result, err := renewScript.Run(ctx, l.rdb, []string{l.key}, l.value, ttlMillis).Int64()
	if err != nil {
		return fmt.Errorf("redis renew script: %w", err)
	}

	if result == 0 {
		return fmt.Errorf("lock not owned or already expired")
	}

	return nil
}

func (l *RedisLock) StartRenew(ctx context.Context, interval time.Duration) context.CancelFunc {
	renewCtx, cancel := context.WithCancel(ctx)

	go func() {
		ticker := time.NewTicker(interval)
		defer ticker.Stop()

		for {
			select {
			case <-renewCtx.Done():
				return
			case <-ticker.C:
				if err := l.Renew(renewCtx); err != nil {
					// 生产里应记录日志和告警
					return
				}
			}
		}
	}()

	return cancel
}
```

---

## 【文本块 44】看门狗会不会导致死锁？

如果客户端正常运行，看门狗会续期。

如果客户端宕机，看门狗 goroutine 也会停止。Redis 中锁的 TTL 不再续期，到期后自动释放。

所以看门狗不会让锁永久存在。

但如果业务 goroutine 卡死，而进程还活着，看门狗可能持续续期，导致锁长时间不释放。生产里应该配合：

* 最大续期次数
* 最大持锁时间
* context deadline
* 业务超时控制
* 监控告警

---

# 十七、Redis 事务

## 【文本块 45】Q：Redis 支持事务吗？

Redis 支持事务，但它和 MySQL 事务不是一回事。

Redis 事务通过：

```text
MULTI
EXEC
DISCARD
WATCH
```

实现。

它能保证一组命令按顺序执行，中间不会插入其他客户端命令。

但 Redis 事务不支持 MySQL 那种完整回滚。

如果 EXEC 前命令语法就错了，整个事务可能不执行。
如果 EXEC 执行过程中某条命令因为类型错误失败，其他命令不会自动回滚。

所以 Redis 事务更像是“命令批量顺序执行”，不是完整 ACID 事务。

---

## 【文本块 46】Q：Redis 为什么不支持完整回滚？

这是设计取舍。

Redis 追求简单和高性能。如果要支持数据库那种完整回滚，就需要维护 undo log、事务隔离、复杂状态管理，会增加很多开销。

Redis 的定位不是强事务数据库，而是高性能内存数据结构服务。

面试回答：

> Redis 事务保证命令排队和顺序执行，但不保证完整回滚。它的设计目标是简单高性能，而不是关系型数据库那种 ACID 事务。

---

## 【代码块 18】go-redis TxPipeline

```go
package main

import (
	"context"
	"fmt"

	"github.com/redis/go-redis/v9"
)

func IncrementTwoKeys(ctx context.Context, rdb *redis.Client) error {
	pipe := rdb.TxPipeline()

	incrA := pipe.Incr(ctx, "counter:a")
	incrB := pipe.Incr(ctx, "counter:b")

	_, err := pipe.Exec(ctx)
	if err != nil {
		return fmt.Errorf("exec tx pipeline: %w", err)
	}

	fmt.Println("a:", incrA.Val())
	fmt.Println("b:", incrB.Val())

	return nil
}
```

---

## 【文本块 47】Pipeline 和事务有什么区别？

Pipeline 是把多个命令一次性发给 Redis，减少网络 RTT。它不保证事务语义。

事务通过 MULTI/EXEC 包裹，保证命令按顺序执行，中间不会插入其他命令。

TxPipeline 是 go-redis 提供的封装，会用 MULTI/EXEC 执行 pipeline 中的命令。

面试回答：

> Pipeline 主要是性能优化，减少网络往返；事务主要是执行语义，保证一组命令连续执行。TxPipeline 则是把两者结合起来。

---

# 十八、Redis 线程模型

## 【文本块 48】Q：Redis 是单线程吗？

准确说，Redis 的核心命令执行主要是单线程，但 Redis 进程不是所有事情都只有一个线程。

比如：

* RDB 持久化会 fork 子进程
* AOF rewrite 会 fork 子进程
* lazy free 可以后台线程释放内存
* Redis 6 以后网络 IO 可以多线程处理
* cluster、异步任务也可能涉及额外线程

所以面试不能简单说“Redis 是单线程”。

标准说法：

> Redis 核心命令执行是单线程的，这样可以避免复杂的锁竞争和并发控制。但 Redis 整个进程并不只有一个线程，持久化、异步释放、网络 IO 等模块可能使用子进程或多线程。

---

## 【文本块 49】Q：Redis 为什么这么快？

Redis 快主要有几个原因：

第一，纯内存操作。
大部分数据都在内存中，避免磁盘随机 IO。

第二，高效数据结构。
SDS、dict、skiplist、quicklist、listpack、intset 等都针对性能和内存做了优化。

第三，单线程命令执行。
避免多线程访问共享数据带来的锁竞争、上下文切换和同步复杂度。

第四，IO 多路复用。
Redis 使用 epoll/kqueue 等机制，一个线程可以处理大量连接事件。

第五，协议简单。
RESP 协议解析成本低。

第六，Redis 6 引入多线程网络 IO。
在高并发大流量场景下，可以并行处理部分网络读写。

一句话：

> Redis 快不是因为单线程本身，而是内存操作、高效数据结构、少锁、IO 多路复用共同作用的结果。

---

## 【文本块 50】Q：Redis 6 为什么引入多线程？

Redis 6 的多线程主要用于网络 IO，不是把核心命令执行改成多线程。

原因是随着机器性能提升，Redis 的瓶颈在某些场景下从命令执行变成了网络收发和协议解析。

所以 Redis 6 把网络 IO 的部分工作多线程化，提高吞吐。

但核心命令执行仍然尽量保持单线程，避免共享数据结构上的复杂锁竞争。

面试回答：

> Redis 6 多线程主要优化网络 IO，而不是让命令执行全面并行。这样既提升了高并发网络收发能力，又保留了单线程命令执行简单、少锁的优势。

---

## 【文本块 51】Q：Redis 是 AP 还是 CP？

默认 Redis 主从、哨兵架构更偏 AP。

原因：

* 主从复制通常是异步的
* 主节点写成功后，从节点可能还没同步
* 主从切换时可能丢失少量数据
* Redis 更强调高可用和低延迟

但不能绝对地说 Redis 永远是 AP。通过 WAIT 命令、同步策略、业务约束，可以在局部增强一致性，但代价是性能和可用性下降。

面试回答：

> 默认 Redis 更偏 AP，因为它优先保证可用性和性能，主从复制又是异步的。但在特定场景下可以通过 WAIT、业务幂等、数据库兜底等方式增强一致性，不能把 Redis 当成强一致数据库使用。

---

# 十九、BigKey

## 【文本块 52】Q：什么是 BigKey？有什么危害？

BigKey 不是 key 名字很长，而是 key 对应的 value 很大。

常见判断标准：

String value 超过 10KB、100KB，甚至 MB 级，就要警惕。
Hash/List/Set/ZSet 元素数量超过几千、几万，也要警惕。

BigKey 危害：

第一，阻塞 Redis 主线程。
比如 DEL 一个巨大 Hash，内存释放可能阻塞主线程。

第二，网络阻塞。
GET 一个几 MB 的 value，会占用大量带宽，拖慢 Redis 和客户端。

第三，内存不均。
Cluster 中某个 slot 里有大 key，会导致节点内存倾斜。

第四，迁移困难。
主从同步、RDB、AOF rewrite、Cluster slot 迁移都会受影响。

第五，客户端超时。
Go 服务读取大 value，可能导致反序列化慢、GC 压力大、请求超时。

---

## 【文本块 53】如何发现 BigKey？

常见方式：

第一，redis-cli --bigkeys。
可以在线扫描大 key，但生产执行要谨慎。

第二，SCAN + 类型判断 + 分批统计。
自己写脚本扫描 key，按类型统计长度或内存。

第三，MEMORY USAGE。
查看 key 具体内存占用。

第四，离线分析 RDB。
用 rdb tools 或云厂商分析工具，不影响线上。

第五，监控慢日志和网络流量。
如果某些命令耗时高、出网流量异常，也可能是 BigKey。

---

## 【文本块 54】如何处理 BigKey？

第一，拆分。
一个大 Hash 拆成多个小 Hash。

例如：

```text
user:relation:{userID}:0
user:relation:{userID}:1
...
```

按好友 ID hash 分片。

第二，分页读取。
不要一次性 HGETALL、SMEMBERS、LRANGE 全量读取大集合。用 HSCAN、SSCAN、ZSCAN 分批。

第三，异步删除。
Redis 4.0 之后，用 UNLINK 替代 DEL 删除大 key，把内存释放交给后台线程。

第四，限制 value 大小。
应用层限制单个缓存对象大小，超过阈值不缓存或拆分缓存。

第五，压缩或换存储。
如果 value 确实大，可以考虑压缩、对象存储、数据库分页，而不是硬塞 Redis。

---

## 【代码块 19】Go 使用 SCAN 分批扫描 key

```go
package main

import (
	"context"
	"fmt"

	"github.com/redis/go-redis/v9"
)

func ScanKeys(ctx context.Context, rdb *redis.Client, pattern string) error {
	var cursor uint64

	for {
		keys, nextCursor, err := rdb.Scan(ctx, cursor, pattern, 100).Result()
		if err != nil {
			return fmt.Errorf("scan keys: %w", err)
		}

		for _, key := range keys {
			fmt.Println("key:", key)
		}

		cursor = nextCursor
		if cursor == 0 {
			break
		}
	}

	return nil
}
```

---

## 【代码块 20】使用 UNLINK 异步删除 key

```go
package main

import (
	"context"
	"fmt"

	"github.com/redis/go-redis/v9"
)

func DeleteBigKey(ctx context.Context, rdb *redis.Client, key string) error {
	if err := rdb.Unlink(ctx, key).Err(); err != nil {
		return fmt.Errorf("unlink big key: %w", err)
	}
	return nil
}
```

---

# 二十、HotKey

## 【文本块 55】Q：什么是 HotKey？有什么危害？

HotKey 指的是某个 key 被极高频访问。

比如：

* 秒杀商品库存 key
* 热搜榜 key
* 明星新闻 key
* 爆款商品详情 key
* 首页推荐配置 key

HotKey 危害：

第一，单 Redis 节点压力过大。
即使是 Cluster，一个 key 也只能落到一个 slot、一个节点上。

第二，网络带宽打满。
大量请求集中读取同一个 key，会造成 Redis 出网流量高。

第三，Go 服务等待 Redis。
大量 goroutine 卡在 Redis 调用上，连接池耗尽，请求延迟上升。

第四，缓存击穿风险高。
HotKey 一旦过期，大量请求打到 DB。

---

## 【文本块 56】如何发现 HotKey？

方式：

1. Redis 监控命令统计
2. 客户端埋点统计 key 访问频率
3. 代理层统计
4. 云厂商 HotKey 分析
5. 慢日志和大流量 key 排查
6. Go 服务侧 Prometheus 统计缓存访问 key pattern

工程上不建议直接把完整 key 全量打到指标里，否则高基数会把 Prometheus 打爆。可以统计 key pattern，比如：

```text
user:{id}
product:{id}
rank:daily
```

---

## 【文本块 57】HotKey 怎么解决？

第一，本地缓存。
在 Go 服务本地使用 Ristretto、BigCache、FreeCache 或简单 LRU，减少 Redis 压力。

第二，singleflight。
合并同进程内对同一个 key 的并发请求。

第三，热点 key 逻辑过期。
避免热点 key 物理过期导致击穿。

第四，多副本。
把热点数据复制成多个 key，分散到多个 Redis 节点，客户端随机读。

例如：

```text
hot:product:1001:0
hot:product:1001:1
hot:product:1001:2
```

第五，限流降级。
热点异常时返回旧值、默认值或限流。

第六，预热。
活动前提前加载热点数据。

---

# 二十一、Pipeline

## 【文本块 58】Q：Redis Pipeline 有什么用？

Pipeline 用于把多个 Redis 命令一次性发给服务端，减少网络 RTT。

如果没有 Pipeline：

```text
请求1 -> 响应1
请求2 -> 响应2
请求3 -> 响应3
```

有 Pipeline：

```text
请求1 请求2 请求3 -> 响应1 响应2 响应3
```

Pipeline 不会减少 Redis 执行命令的 CPU 时间，但可以显著减少网络往返开销。

适合：

* 批量 Get
* 批量 Set
* 批量删除
* 批量统计

不适合：

* 每条命令依赖上一条命令结果的复杂逻辑
* 超大批量一次性 pipeline，可能造成 Redis 和客户端内存压力

---

## 【代码块 21】go-redis Pipeline 批量 Get

```go
package main

import (
	"context"
	"fmt"

	"github.com/redis/go-redis/v9"
)

func BatchGet(ctx context.Context, rdb *redis.Client, keys []string) (map[string]string, error) {
	pipe := rdb.Pipeline()

	cmds := make(map[string]*redis.StringCmd, len(keys))
	for _, key := range keys {
		cmds[key] = pipe.Get(ctx, key)
	}

	_, err := pipe.Exec(ctx)
	if err != nil && err != redis.Nil {
		return nil, fmt.Errorf("pipeline exec: %w", err)
	}

	result := make(map[string]string, len(keys))
	for key, cmd := range cmds {
		val, err := cmd.Result()
		if err == redis.Nil {
			continue
		}
		if err != nil {
			return nil, fmt.Errorf("get key %s: %w", key, err)
		}
		result[key] = val
	}

	return result, nil
}
```

---

## 【文本块 59】Pipeline 和 MGET 怎么选？

如果只是批量获取多个 String key，MGET 更简单。

如果是不同类型、不同命令组合，比如 GET + HGET + ZSCORE，Pipeline 更灵活。

Cluster 场景要注意：

* MGET 多 key 如果跨 slot 会受限
* Pipeline 在 cluster client 中会按节点拆分发送
* 多 key 操作最好设计 hash tag 保证同槽

例如：

```text
user:{1001}:profile
user:{1001}:setting
```

花括号里的内容用于计算 slot。

---

# 二十二、Redis Go 工程最佳实践

## 【文本块 60】Go 项目使用 Redis 要注意什么？

第一，所有 Redis 调用都带 context。
不要使用永不超时的 context.Background() 直接贯穿业务请求。

第二，区分 redis.Nil 和真实错误。
key 不存在是业务分支，不是系统错误。

第三，设置合理连接池。
PoolSize 太小会等待连接，太大会压垮 Redis。

第四，设置读写超时。
不要让 Redis 抖动拖死 Go 服务。

第五，避免大 key。
不要随意 HGETALL、SMEMBERS、LRANGE 全量拉大集合。

第六，热点 key 要有保护。
本地缓存、singleflight、逻辑过期、随机 TTL。

第七，缓存要有降级方案。
Redis 不可用时，是读 DB、读本地旧值、返回默认值，还是直接失败，要提前设计。

第八，缓存错误不要都吞。
有些非核心缓存可以降级，但核心流程要记录日志和指标。

第九，key 命名要规范。
方便排查、扫描、治理和批量删除。

第十，指标要完善。
包括命中率、耗时、错误率、连接池等待、Redis 慢查询、BigKey、HotKey。

---

## 【代码块 22】封装 Redis Get：区分 nil 和错误

```go
package main

import (
	"context"
	"fmt"

	"github.com/redis/go-redis/v9"
)

func RedisGetString(ctx context.Context, rdb *redis.Client, key string) (string, bool, error) {
	val, err := rdb.Get(ctx, key).Result()
	if err == redis.Nil {
		return "", false, nil
	}
	if err != nil {
		return "", false, fmt.Errorf("redis get %s: %w", key, err)
	}

	return val, true, nil
}
```

---

## 【文本块 61】Redis 错误要不要影响主流程？

这要看缓存定位。

如果 Redis 是旁路缓存，数据库才是主存储，那么 Redis 读取失败时可以降级查 DB，但要记录日志和指标。

如果 Redis 存的是分布式锁、限流计数、验证码、session、库存预扣等核心状态，Redis 错误就不能简单忽略。

面试可以这样回答：

> Redis 错误是否影响主流程，取决于它在业务里的角色。如果只是旁路缓存，读失败可以降级 DB；如果承载锁、限流、验证码、库存等关键语义，就必须失败返回或走专门兜底，不能静默忽略。

---

# 二十三、本章高频面试速记

## 【文本块 62】数据类型速记

String：对象缓存、计数器、锁、token，底层 SDS。
Hash：对象字段，适合局部更新，小数据 listpack，大数据 hashtable。
List：简单队列、最近记录，底层 quicklist/listpack。
Set：去重、交并差、标签、抽奖，小整数可能 intset，大数据 hashtable。
ZSet：排行榜、延时队列，底层 skiplist + hashtable。
Bitmap：签到、在线状态、二值统计。
HyperLogLog：大规模 UV 近似统计。
Stream：Redis 轻量消息流。

---

## 【文本块 63】高可用速记

RDB：快照，恢复快，可能丢快照间隔内数据。
AOF：追加日志，数据更安全，文件更大，恢复较慢。
混合持久化：兼顾恢复速度和安全性。
主从复制：读扩展和副本。
哨兵：自动故障转移。
Cluster：16384 槽，分片扩容。
Redis 默认更偏 AP，不是强一致数据库。

---

## 【文本块 64】缓存问题速记

缓存穿透：查不存在数据。
解决：缓存空对象、布隆过滤器。

缓存击穿：热点 key 过期。
解决：互斥锁、逻辑过期、singleflight、热点预热。

缓存雪崩：大量 key 同时失效或 Redis 整体不可用。
解决：随机 TTL、预热、高可用、本地缓存、限流熔断降级。

---

## 【文本块 65】双写一致性速记

一般推荐：先更新数据库，再删除缓存。

删除优于更新，因为更新缓存容易浪费，也容易并发乱序。

先删缓存再更新 DB 容易让并发读写回旧缓存。

先更新 DB 再删缓存也有删除失败风险，需要补偿。

补偿方式：

* 重试
* MQ
* binlog 监听
* outbox pattern
* TTL 兜底

强一致场景不要过度依赖缓存，应以数据库事务、锁、唯一约束、幂等为核心。

---

## 【文本块 66】分布式锁速记

加锁：

```text
SET key value NX EX seconds
```

不能 SETNX 后 EXPIRE，因为不是原子操作。

value 必须唯一，防止误删别人的锁。

解锁必须 Lua：

```lua
if redis.call("get", KEYS[1]) == ARGV[1] then
    return redis.call("del", KEYS[1])
else
    return 0
end
```

业务时间超过 TTL 需要续期，看门狗本质是定时检查并延长 TTL。

Redis 锁不是强一致锁，不能替代数据库事务和业务幂等。

---

## 【文本块 67】线程模型速记

Redis 核心命令执行主要是单线程。
Redis 整个进程不是只有一个线程。
Redis 快来自内存、高效数据结构、单线程少锁、IO 多路复用。
Redis 6 多线程主要优化网络 IO，不是全面多线程执行命令。
Redis 默认主从复制异步，更偏 AP。

---

## 【文本块 68】BigKey/HotKey 速记

BigKey 是 value 大，不是 key 名字长。
危害：阻塞主线程、网络阻塞、内存倾斜、迁移困难、客户端超时。
发现：bigkeys、SCAN、MEMORY USAGE、RDB 分析、监控。
处理：拆分、分页、UNLINK、限制大小、压缩或换存储。

HotKey 是访问频率极高的 key。
危害：单节点压力、带宽打满、连接池耗尽、击穿风险。
处理：本地缓存、singleflight、逻辑过期、多副本、限流降级、预热。

---

# 二十四、本章最终面试回答模板

## 【文本块 69】Redis 综合回答模板

如果面试官让我系统讲 Redis，我会按“数据结构、性能、高可用、缓存问题、一致性、Go 落地”这条线来讲。

Redis 常用数据类型包括 String、Hash、List、Set、ZSet。String 底层是 SDS，适合对象缓存、计数器和锁；Hash 适合对象字段级更新；List 可以做简单队列和最近记录；Set 适合去重和集合运算；ZSet 底层大数据量时是 skiplist + hashtable，适合排行榜和延时队列。除此之外，Bitmap 适合签到这类二值统计，HyperLogLog 适合大规模 UV 近似统计。

Redis 快的原因不是单线程一个点，而是纯内存操作、高效数据结构、核心命令单线程少锁、IO 多路复用共同作用。Redis 6 之后引入多线程，主要是优化网络 IO，不是把命令执行改成全面多线程。

Redis 高可用可以按四层理解：RDB/AOF 解决数据恢复，主从复制解决读扩展和副本，哨兵解决主节点故障自动切换，Cluster 通过 16384 个槽解决容量和写入扩展。默认 Redis 主从复制是异步的，所以更偏 AP，不适合作为强一致数据库。

缓存常见问题有穿透、击穿、雪崩。穿透是查不存在的数据，可以用缓存空对象或布隆过滤器；击穿是热点 key 过期，可以用互斥锁、逻辑过期和 Go 里的 singleflight；雪崩是大量 key 同时失效或 Redis 整体不可用，可以用随机 TTL、预热、高可用、本地缓存和限流降级。

缓存和数据库一致性方面，一般推荐先更新数据库，再删除缓存，而不是更新缓存。因为更新缓存有并发乱序和资源浪费问题。先更新 DB 再删缓存也有删除失败风险，所以生产中通常会加 MQ 重试、binlog 监听、outbox 或 TTL 兜底。

分布式锁方面，正确加锁应该使用 `SET key value NX EX seconds`，value 必须是唯一标识，释放锁必须用 Lua 脚本保证判断 value 和删除 key 的原子性。如果业务时间可能超过 TTL，需要续期机制。但 Redis 锁不是强一致锁，不能替代数据库事务、唯一约束和业务幂等。

在 Go 项目里，我会使用 go-redis，并且所有 Redis 调用都带 context timeout，合理配置连接池和读写超时。缓存读取要区分 redis.Nil 和真实错误。热点 key 会结合本地缓存、singleflight、逻辑过期和随机 TTL 做保护。对于 BigKey，要避免一次性 HGETALL、SMEMBERS、LRANGE 全量读取，大 key 删除用 UNLINK，集合型大 key 要拆分。

一句话总结：

Redis 面试的核心不是会用 Get/Set，而是要讲清楚它为什么快、数据结构怎么选、高可用怎么做、缓存异常怎么防、双写一致性怎么兜底，以及在 Go 服务里如何用 context、连接池、singleflight、Lua 和降级机制把它稳定地落地到生产系统中。
